In [1]:
# If you haven't set your access token as an environment variable, pass it in here.
from getpass import getpass

ACCESS_TOKEN = getpass()

In [2]:
from langchain_community.document_loaders import GithubFileLoader


In [3]:
loader = GithubFileLoader(
    repo="anirban2005143a/Atom-Accounting",  # the repo name
    branch="main",  # the branch name
    access_token=ACCESS_TOKEN,
    github_api_url="https://api.github.com",
    file_filter= lambda path: "frontend/src/context/company/CompanyContext.jsx" in path
)
documents = loader.load()

In [5]:
documents[0].page_content

'import {\n  createContext,\n  useCallback,\n  useContext,\n  useEffect,\n  useReducer,\n  useState,\n} from "react";\nimport axios from "axios";\nimport { showToast } from "../../utils/showToast";\nimport { useNavigate } from "react-router-dom";\nimport { LogoAnimation } from "../../component/logo-animation";\nimport { OnBoardingPage } from "../../routes/onBoarding/OnBoarding";\nimport { validateFields } from "../../utils/checkFormValidation";\nimport { UserContext } from "../userContext/UserContext";\nimport { uploadFile } from "../../utils/uploadFiles";\n\nexport const CompanyContext = createContext();\n\nexport const initialState = {\n  companyName: "",\n  companyLogo: "",\n  companyStreet: "",\n  companyZIP: "",\n  companyState: "",\n  companyLandmark: "",\n  companyMobileNo: "",\n  companyWebsite: "",\n  companyGSTIN: "",\n  companyPAN: "",\n  companyEmail: "",\n  companyCIN: "",\n};\n\nexport function formReducer(state, action) {\n  switch (action.type) {\n    case "UPDATE_FIELD

In [40]:
from pathlib import Path
Path(documents[0].metadata['source']).suffix

'.jsx'

# parse code

In [48]:
from tree_sitter import Parser
from tree_sitter_languages import get_language, get_parser

EXTENSION_TO_LANGUAGE = {
    ".py": "python",
    ".js": "javascript",
    ".jsx": "javascript",
    ".ts": "typescript",
    ".tsx": "typescript",
    ".cpp": "cpp",
    ".c": "c",
    ".java": "java",
}


def parse_code(code: str, file_extension: str):
    language_name = EXTENSION_TO_LANGUAGE.get(file_extension)

    if language_name is None:
        raise ValueError(f"Unsupported file extension: {file_extension}")

    parser = get_parser("javascript")
    tree = parser.parse(code.encode("utf-8"))

    return tree


In [8]:
from tree_sitter import Parser
from tree_sitter_language_pack import get_language , get_parser

# Example: load JavaScript (JSX works here too)
language = get_language("javascript")  # or "tsx" for TSX
parser = get_parser("javascript")

# Example code
code = """
import React from 'react';

export default function App() {
    return <h1>Hello World</h1>;
}
"""

tree = parser.parse(bytes(code, "utf8"))
print(tree.root_node)  # prints root node


(program (import_statement (import_clause (identifier)) source: (string (string_fragment))) (export_statement declaration: (function_declaration name: (identifier) parameters: (formal_parameters) body: (statement_block (return_statement (jsx_element open_tag: (jsx_opening_element name: (identifier)) (jsx_text) close_tag: (jsx_closing_element name: (identifier))))))))


In [9]:
def walk(node, source_code, results=None):
    if results is None:
        results = []

    results.append({
        "type": node.type,
        "start": node.start_point,
        "end": node.end_point,
        "text": source_code[node.start_byte:node.end_byte]
    })

    for child in node.children:
        walk(child, source_code, results)

    return results

nodes = walk(tree.root_node, code)
for n in nodes:
    print(n["type"], n["start"], n["end"])


program Point(row=1, column=0) Point(row=6, column=0)
import_statement Point(row=1, column=0) Point(row=1, column=26)
import Point(row=1, column=0) Point(row=1, column=6)
import_clause Point(row=1, column=7) Point(row=1, column=12)
identifier Point(row=1, column=7) Point(row=1, column=12)
from Point(row=1, column=13) Point(row=1, column=17)
string Point(row=1, column=18) Point(row=1, column=25)
' Point(row=1, column=18) Point(row=1, column=19)
string_fragment Point(row=1, column=19) Point(row=1, column=24)
' Point(row=1, column=24) Point(row=1, column=25)
; Point(row=1, column=25) Point(row=1, column=26)
export_statement Point(row=3, column=0) Point(row=5, column=1)
export Point(row=3, column=0) Point(row=3, column=6)
default Point(row=3, column=7) Point(row=3, column=14)
function_declaration Point(row=3, column=15) Point(row=5, column=1)
function Point(row=3, column=15) Point(row=3, column=23)
identifier Point(row=3, column=24) Point(row=3, column=27)
formal_parameters Point(row=3, co

In [12]:
for n in nodes:
    print(n["type"])

program
import_statement
import
import_clause
identifier
from
string
'
string_fragment
'
;
export_statement
export
default
function_declaration
function
identifier
formal_parameters
(
)
statement_block
{
return_statement
return
jsx_element
jsx_opening_element
<
identifier
>
jsx_text
jsx_closing_element
</
identifier
>
;
}


In [10]:
tree.root_node

<Node type=program, start_point=(1, 0), end_point=(6, 0)>

In [ ]:
code = documents[0].page_content

tree = parser.parse(bytes(code, "utf8"))
print(tree.root_node)

def extract_chunks(node, code, results=None):
    if results is None:
        results = []
    
    chunk = {
        "type": node.type,
        "code": code[node.start_byte:node.end_byte],
        "start_line": node.start_point[0],
        "end_line": node.end_point[0]
    }
    results.append(chunk)

    # Visit children recursively
    for child in node.children:
        extract_chunks(child, code, results)

    return results

nodes = extract_chunks(tree.root_node, code)
for n in nodes:
    print(n)

(program (import_statement (import_clause (named_imports (import_specifier name: (identifier)) (import_specifier name: (identifier)) (import_specifier name: (identifier)) (import_specifier name: (identifier)) (import_specifier name: (identifier)) (import_specifier name: (identifier)))) source: (string (string_fragment))) (import_statement (import_clause (identifier)) source: (string (string_fragment))) (import_statement (import_clause (named_imports (import_specifier name: (identifier)))) source: (string (string_fragment))) (import_statement (import_clause (named_imports (import_specifier name: (identifier)))) source: (string (string_fragment))) (import_statement (import_clause (named_imports (import_specifier name: (identifier)))) source: (string (string_fragment))) (import_statement (import_clause (named_imports (import_specifier name: (identifier)))) source: (string (string_fragment))) (import_statement (import_clause (named_imports (import_specifier name: (identifier)))) source: (s

In [18]:
for n in nodes:
    print(n['type'])

program
import_statement
import
import_clause
named_imports
{
import_specifier
identifier
,
import_specifier
identifier
,
import_specifier
identifier
,
import_specifier
identifier
,
import_specifier
identifier
,
import_specifier
identifier
,
}
from
string
"
string_fragment
"
;
import_statement
import
import_clause
identifier
from
string
"
string_fragment
"
;
import_statement
import
import_clause
named_imports
{
import_specifier
identifier
}
from
string
"
string_fragment
"
;
import_statement
import
import_clause
named_imports
{
import_specifier
identifier
}
from
string
"
string_fragment
"
;
import_statement
import
import_clause
named_imports
{
import_specifier
identifier
}
from
string
"
string_fragment
"
;
import_statement
import
import_clause
named_imports
{
import_specifier
identifier
}
from
string
"
string_fragment
"
;
import_statement
import
import_clause
named_imports
{
import_specifier
identifier
}
from
string
"
string_fragment
"
;
import_statement
import
import_clause
named_impor

# all important node type 

In [ ]:
target_types = [
    # --- Existing Targets ---
    "function_definition", "function_declaration", "method_definition",
    "class_definition", "class_declaration", "arrow_function", 
    "if_statement", "for_statement", "while_statement", "try_statement", "jsx_element",

    # --- Type & Interface Definitions (Crucial for TS/Go/Java/C#) ---
    "interface_declaration", "type_alias_declaration", "enum_declaration",
    "struct_specifier", "type_definition",

    # --- Advanced Logic & Data Structure ---
    "switch_statement", "case_clause", "catch_clause", 
    "with_statement", "do_statement",

    # --- Modern JS/TS & Export Logic ---
    "export_statement", "lexical_declaration", # Captures 'const/let' at top level
    "jsx_self_closing_element", "pair", # Captures key-value pairs in large config objects/JSON

    # --- System & Lifecycle ---
    "unit_test", "decorator", # Captures @decorators in Python/TS
    "preproc_def", # C/C++ Macros
    "namespace_definition" # C++/C# Namespaces
]

In [ ]:
IMPORTANT_NODE_TYPES = [
    # --- Definitions & Structure ---
    "function_definition", "function_declaration", "method_definition",
    "class_definition", "class_declaration", "interface_declaration",
    "module", "package_clause", "struct_specifier", "enum_declaration",
    
    # --- Logic & Control Flow ---
    "if_statement", "else_clause", "elif_clause",
    "for_statement", "while_statement", "do_statement", "for_in_statement",
    "switch_statement", "case_statement", "try_statement", "catch_clause",
    "finally_clause", "throw_statement", "return_statement", "break_statement", "continue_statement",

    # --- Data & Assignments ---
    "variable_declaration", "variable_declarator", "assignment_expression",
    "augmented_assignment_expression", "lexical_declaration", # JS specific
    "formal_parameters", "parameter", "identifier",
    
    # --- Expressions & Calls ---
    "call_expression", "argument_list", "binary_expression", 
    "unary_expression", "update_expression", "lambda", "arrow_function",
    "member_expression", "attribute", "subscript",
    
    # --- Imports & Exports ---
    "import_statement", "import_from_statement", "export_statement",
    "require_call", "use_declaration",
    
    # --- Web / UI Specific ---
    "jsx_element", "jsx_self_closing_element", "template_string",
    
    # --- Comments (Critical for Documentation Tools) ---
    "comment", "line_comment", "block_comment"
]

In [ ]:
MASTER_TARGET_TYPES = [
    # --- 1. Top-Level Definitions & Modules ---
    "module", "package_clause", "namespace_definition", "program",
    "import_statement", "import_from_statement", "export_statement", "require_call", 
    "use_declaration", "extern_crate_declaration", # Rust specific

    # --- 2. Classes, Interfaces & Types ---
    "class_definition", "class_declaration", "interface_declaration", 
    "struct_specifier", "type_definition", "type_alias_declaration", 
    "enum_declaration", "protocol_declaration", # Swift specific
    "trait_declaration", # Rust specific
    "impl_item", # Rust implementations

    # --- 3. Functions & Methods ---
    "function_definition", "function_declaration", "method_definition", 
    "method_declaration", "arrow_function", "lambda", "generator_function",
    "function_item", "parameter", "formal_parameters",

    # --- 4. Logic & Control Flow ---
    "if_statement", "else_clause", "elif_clause", "switch_statement", 
    "case_clause", "case_statement", "match_statement", "match_arm", # Python/Rust match
    "for_statement", "for_in_statement", "while_statement", "do_statement", 
    "try_statement", "catch_clause", "finally_clause", "except_clause", # Python specific
    "throw_statement", "return_statement", "break_statement", "continue_statement",
    "with_statement", "using_statement", # Python/C# context managers

    # --- 5. Data, Variables & Objects ---
    "variable_declaration", "variable_declarator", "lexical_declaration", 
    "assignment_expression", "augmented_assignment_expression",
    "pair", "dictionary", "list", "array", # Data structure literals
    "object_pattern", "array_pattern", # Destructuring context

    # --- 6. Expressions & Execution ---
    "call_expression", "argument_list", "binary_expression", 
    "unary_expression", "update_expression", "member_expression", 
    "attribute", "subscript", "await_expression", "yield_expression",
    "go_statement", "defer_statement", # Go specific concurrency

    # --- 7. Web, UI & Templating ---
    "jsx_element", "jsx_self_closing_element", "jsx_opening_element", 
    "template_string", "template_substitution",
    "html_element", "attribute_name", "attribute_value",

    # --- 8. Metadata & Documentation ---
    "comment", "line_comment", "block_comment", 
    "decorator", "attribute_item", # Python @dec or Rust #[attr]
    "preproc_def", "preproc_include", # C/C++ macros
    "unit_test", "test_definition"
]

In [ ]:
JS_TS_TARGETS = [
    "export_statement", "import_statement", "lexical_declaration", 
    "variable_declarator", "arrow_function", "function_declaration", 
    "method_definition", "class_declaration", "interface_declaration", 
    "type_alias_declaration", "enum_declaration", "jsx_element", 
    "jsx_self_closing_element", "try_statement", "catch_clause", 
    "if_statement", "switch_statement", "for_statement", "while_statement",
    "template_substitution", "call_expression"
]

PYTHON_TARGETS = [
    "function_definition", "class_definition", "decorated_definition", 
    "if_statement", "elif_clause", "else_clause", "for_statement", 
    "while_statement", "try_statement", "except_clause", "with_statement", 
    "lambda", "assignment", "call", "import_from_statement", 
    "import_statement", "decorator"
]

SYSTEMS_TARGETS = [
    # Rust
    "struct_item", "enum_item", "trait_item", "impl_item", "function_item", 
    "match_statement", "match_arm", "let_declaration", "macro_definition",
    # Go
    "type_declaration", "method_declaration", "func_declaration", 
    "go_statement", "defer_statement", "select_statement", "communication_case",
    # C++
    "namespace_definition", "template_declaration", "function_definition", 
    "class_specifier", "struct_specifier", "preproc_def"
]

ENTERPRISE_TARGETS = [
    "class_declaration", "interface_declaration", "enum_declaration", 
    "method_declaration", "constructor_declaration", "property_declaration", 
    "protocol_declaration", "extension_declaration", "variable_declaration", 
    "if_expression", "when_expression", "try_expression", "lambda_expression"
]

LANGUAGE_MAP = {
    ".py": PYTHON_TARGETS,
    ".js": JS_TS_TARGETS,
    ".ts": JS_TS_TARGETS,
    ".tsx": JS_TS_TARGETS,
    ".go": SYSTEMS_TARGETS,
    ".rs": SYSTEMS_TARGETS,
    ".cpp": SYSTEMS_TARGETS,
    ".java": ENTERPRISE_TARGETS
}


# extract all metadata and important info from node

In [19]:
def get_metadata(node, source_code, file_path):
    metadata = {
        "node_type": node.type,
        "file_path": file_path,
        "name": None,
        "signature": None,
        "dependencies": []
    }

    # 1. Extract Name & Signature
    # Most definitions have an 'identifier' child that represents the name
    for child in node.children:
        if child.type == "identifier":
            metadata["name"] = source_code[child.start_byte:child.end_byte].decode("utf-8")
        
        # The 'signature' is usually the start of the node until the block/body begins
        if child.type in ["block", "statement_block", "compound_statement"]:
            sig_end = child.start_byte
            metadata["signature"] = source_code[node.start_byte:sig_end].decode("utf-8").strip()

    # 2. Extract Dependencies (Function Calls inside this node)
    # We look for all 'call_expression' nodes nested within this function/class
    def find_calls(n):
        calls = []
        for c in n.children:
            if c.type == "call_expression":
                # The first child of a call_expression is usually the identifier being called
                function_called = source_code[c.children[0].start_byte:c.children[0].end_byte].decode("utf-8")
                calls.append(function_called)
            calls.extend(find_calls(c))
        return list(set(calls)) # Unique calls only

    metadata["dependencies"] = find_calls(node)
    
    return metadata

# updated fucntion with all in one

In [ ]:
def extract_chunks_with_metadata(node, code_bytes, file_path="unknown.py", results=None):
    if results is None:
        results = []

    # Target nodes that usually represent a logical "block" of code
    target_types = [
        "function_definition", "function_declaration", "method_definition",
        "class_definition", "class_declaration", "if_statement", 
        "while_statement", "for_statement", "jsx_element"
    ]

    # Only process "named" nodes to avoid syntax tokens like '(' or ':'
    if node.is_named:
        # 1. Base Info
        chunk_code = code_bytes[node.start_byte:node.end_byte].decode("utf8")
        
        metadata = {
            "node_type": node.type,
            "file_path": file_path,
            "start_line": node.start_point[0],
            "end_line": node.end_point[0],
            "name": "anonymous",  # Default
            "signature": "",
            "dependencies": []
        }

        # 2. Extract Name (Look for 'name' field or 'identifier' child)
        name_node = node.child_by_field_name("name")
        if not name_node:
            for child in node.children:
                if child.type == "identifier":
                    name_node = child
                    break
        
        if name_node:
            metadata["name"] = code_bytes[name_node.start_byte:name_node.end_byte].decode("utf8")

        # 3. Extract Signature (Start of node to start of body/block)
        body_node = node.child_by_field_name("body")
        if not body_node:
            # Fallback for languages where body isn't a named field
            for child in node.children:
                if child.type in ["block", "statement_block", "compound_statement"]:
                    body_node = child
                    break
        
        if body_node:
            sig_bytes = code_bytes[node.start_byte:body_node.start_byte]
            metadata["signature"] = sig_bytes.decode("utf8").strip()
        else:
            # If no body (like a simple statement), use the first line as signature
            metadata["signature"] = chunk_code.split('\n')[0]

        # 4. Extract Dependencies (Find all function calls inside this node)
        def find_calls(n, calls_list):
            for child in n.children:
                if child.type == "call_expression":
                    # Usually the first child of the call is the function name
                    call_id = child.child_by_field_name("function") or child.children[0]
                    call_name = code_bytes[call_id.start_byte:call_id.end_byte].decode("utf8")
                    calls_list.append(call_name)
                find_calls(child, calls_list)

        calls = []
        find_calls(node, calls)
        metadata["dependencies"] = list(set(calls)) # Unique names

        # Add to results
        results.append({
            "code": chunk_code,
            "metadata": metadata
        })

    # Recursive call to visit children
    for child in node.children:
        extract_chunks_with_metadata(child, code_bytes, file_path, results)

    return results

# --- Usage ---
code_bytes = bytes(documents[0].page_content, "utf8")
tree = parser.parse(code_bytes)
all_chunks = extract_chunks_with_metadata(tree.root_node, code_bytes, file_path="my_script.py")



Type: program | Name: anonymous
Sig: import {
Deps: ['createContext', 'res.data.status.toLowerCase', 'useContext', 'companyFormDispatch', 'navigate', 'res.data.data.slice', 'useReducer', 'getCompanyList', 'res.data.data.filter', 'uploadFile', 'Object.keys', 'setisAuthenticating', 'useNavigate', 'showToast', 'res.data.error_message?.toLowerCase', 'setErrors', 'useCallback', 'localStorage.getItem', 'setcompanyList', 'localStorage.getItem("companyid").toString', 'validateFields', 'window.location.reload', 'getCompanyDetails', 'setcompanyDetails', 'res.data.error_message.toLowerCase', 'axios.post', 'useEffect', 'axios.get', 'console.log', 'localStorage.setItem', 'useState', 'setisLoading']
---
Type: import_statement | Name: anonymous
Sig: import {
Deps: []
---
Type: import_clause | Name: anonymous
Sig: {
Deps: []
---


In [25]:
# Print first 3 chunks to verify
for chunk in all_chunks:
    print(f"Type: {chunk['metadata']['node_type']} | Name: {chunk['metadata']['name']}")
    print(f"Sig: {chunk['metadata']['signature']}")
    # print(f"Code: {chunk['code']}")
    print(f"Deps: {chunk['metadata']['dependencies']}\n---")

Type: program | Name: anonymous
Sig: import {
Deps: ['createContext', 'res.data.status.toLowerCase', 'useContext', 'companyFormDispatch', 'navigate', 'res.data.data.slice', 'useReducer', 'getCompanyList', 'res.data.data.filter', 'uploadFile', 'Object.keys', 'setisAuthenticating', 'useNavigate', 'showToast', 'res.data.error_message?.toLowerCase', 'setErrors', 'useCallback', 'localStorage.getItem', 'setcompanyList', 'localStorage.getItem("companyid").toString', 'validateFields', 'window.location.reload', 'getCompanyDetails', 'setcompanyDetails', 'res.data.error_message.toLowerCase', 'axios.post', 'useEffect', 'axios.get', 'console.log', 'localStorage.setItem', 'useState', 'setisLoading']
---
Type: import_statement | Name: anonymous
Sig: import {
Deps: []
---
Type: import_clause | Name: anonymous
Sig: {
Deps: []
---
Type: named_imports | Name: anonymous
Sig: {
Deps: []
---
Type: import_specifier | Name: createContext
Sig: createContext
Deps: []
---
Type: identifier | Name: anonymous
Sig: 

In [32]:
def extract_chunks_with_metadata(node, code_bytes, file_path="unknown.py", results=None):
    if results is None:
        results = []

    # Target nodes for RAG
    target_types = [
        "function_definition", "function_declaration", "method_definition",
        "class_definition", "class_declaration", "arrow_function", 
        "if_statement", "for_statement", "while_statement", "try_statement", "jsx_element"
    ]

    if node.is_named and node.type in target_types:
        # 1. Capture ONLY the code for this specific node
        chunk_code = code_bytes[node.start_byte:node.end_byte].decode("utf8")
        
        metadata = {
            "node_type": node.type,
            "file_path": file_path,
            "start_line": node.start_point[0],
            "end_line": node.end_point[0],
            "name": "anonymous",
            "signature": "",
            "dependencies": []
        }

        # 2. ADVANCED NAMING LOGIC
        # Standard: Look for a 'name' field
        name_node = node.child_by_field_name("name")
        
        # Arrow Function Fix: Look "up" or "sideways" for the variable name
        if node.type == "arrow_function":
            parent = node.parent
            # In JS: const myName = () => {} -> 'myName' is in the variable_declarator
            if parent and parent.type == "variable_declarator":
                id_node = parent.child_by_field_name("name")
                if id_node:
                    name_node = id_node
        
        # Fallback: Search children for identifier if name_node still None
        if not name_node:
            for child in node.children:
                if child.type == "identifier":
                    name_node = child
                    break
        
        if name_node:
            metadata["name"] = code_bytes[name_node.start_byte:name_node.end_byte].decode("utf8")

        # 3. SIGNATURE LOGIC
        # Find the body to exclude it from the signature
        body_node = node.child_by_field_name("body")
        if body_node:
            sig_bytes = code_bytes[node.start_byte:body_node.start_byte]
            metadata["signature"] = sig_bytes.decode("utf8").strip()
        else:
            # For arrow functions or small blocks, take the first line
            metadata["signature"] = chunk_code.split('\n')[0]

        # 4. DEPENDENCY LOGIC (Function calls inside this node)
        def find_calls(n, calls_list):
            for child in n.children:
                if child.type == "call_expression":
                    # Get the 'function' being called
                    call_id = child.child_by_field_name("function") or child.children[0]
                    try:
                        call_name = code_bytes[call_id.start_byte:call_id.end_byte].decode("utf8")
                        calls_list.append(call_name)
                    except: pass
                find_calls(child, calls_list)

        calls = []
        find_calls(node, calls)
        metadata["dependencies"] = list(set(calls))

        results.append({
            "code": chunk_code, # Now contains ONLY the code for this node
            "metadata": metadata
        })

    # Continue traversing the whole tree
    for child in node.children:
        extract_chunks_with_metadata(child, code_bytes, file_path, results)

    return results

In [33]:
all_chunks = extract_chunks_with_metadata(tree.root_node, code_bytes, file_path="my_script.py")


In [34]:
for item in all_chunks:
    meta = item['metadata']
    print(f"--- NODE FOUND ---")
    print(f"Type: {meta['node_type']}")
    print(f"Name: {meta['name']}")
    print(f"Signature: {meta['signature']}")
    print(f"Dependencies: {meta['dependencies']}")
    print(f"Lines: {meta['start_line']} to {meta['end_line']}")
    # print(f"Code Snippet:\n{item['code']}") # Uncomment to see the actual code
    print("\n")

--- NODE FOUND ---
Type: function_declaration
Name: formReducer
Signature: function formReducer(state, action)
Dependencies: []
Lines: 34 to 46


--- NODE FOUND ---
Type: arrow_function
Name: CompanyContextProvider
Signature: ({ children }) =>
Dependencies: ['res.data.status.toLowerCase', 'useContext', 'companyFormDispatch', 'navigate', 'res.data.data.slice', 'useReducer', 'getCompanyList', 'res.data.data.filter', 'uploadFile', 'Object.keys', 'setisAuthenticating', 'useNavigate', 'showToast', 'res.data.error_message?.toLowerCase', 'setErrors', 'useCallback', 'localStorage.getItem', 'setcompanyList', 'localStorage.getItem("companyid").toString', 'validateFields', 'window.location.reload', 'getCompanyDetails', 'setcompanyDetails', 'res.data.error_message.toLowerCase', 'axios.post', 'useEffect', 'axios.get', 'console.log', 'localStorage.setItem', 'useState', 'setisLoading']
Lines: 48 to 411


--- NODE FOUND ---
Type: arrow_function
Name: anonymous
Signature: async (setisLoading = () => {}

In [29]:
def extract_chunks_with_metadata(node, code_bytes, file_path="unknown.py", results=None):
    if results is None:
        results = []

    # Nodes worth chunking for RAG
    target_types = [
        "function_definition", "function_declaration", "method_definition",
        "class_definition", "class_declaration", "arrow_function", 
        "if_statement", "for_statement", "while_statement", "try_statement", "jsx_element"
    ]

    if node.is_named and node.type in target_types:
        # 1. Capture ONLY the code for this specific node
        chunk_code = code_bytes[node.start_byte:node.end_byte].decode("utf8", errors="ignore")
        
        start_line = node.start_point[0] + 1  # 1-indexed for humans
        end_line = node.end_point[0] + 1

        # 2. SMART NAMING LOGIC
        name = "anonymous"
        
        # Try finding standard name (e.g. def my_func)
        name_node = node.child_by_field_name("name")
        
        # Try finding Arrow Function name (const myFunc = ...)
        if not name_node and node.type == "arrow_function":
            parent = node.parent
            if parent and parent.type == "variable_declarator":
                name_node = parent.child_by_field_name("name")

        # Extract name text or generate synthetic name for logic blocks
        if name_node:
            name = code_bytes[name_node.start_byte:name_node.end_byte].decode("utf8", errors="ignore")
        else:
            # For if_statements, etc., generate a name using type + line
            name = f"{node.type}_at_line_{start_line}"

        # 3. SIGNATURE LOGIC
        body_node = node.child_by_field_name("body")
        if body_node:
            sig_bytes = code_bytes[node.start_byte:body_node.start_byte]
            signature = sig_bytes.decode("utf8", errors="ignore").strip()
        else:
            signature = chunk_code.split('\n')[0]

        # 4. DEPENDENCY LOGIC
        def find_calls(n, calls_list):
            for child in n.children:
                if child.type == "call_expression":
                    call_id = child.child_by_field_name("function") or child.children[0]
                    try:
                        c_name = code_bytes[call_id.start_byte:call_id.end_byte].decode("utf8")
                        calls_list.append(c_name)
                    except: pass
                find_calls(child, calls_list)

        calls = []
        find_calls(node, calls)

        # Create the structured object
        results.append({
            "code": chunk_code,  # This is the "code attribute" you requested
            "metadata": {
                "name": name,
                "node_type": node.type,
                "file_path": file_path,
                "signature": signature,
                "dependencies": list(set(calls)),
                "start_line": start_line,
                "end_line": end_line
            }
        })

    # Continue traversal
    for child in node.children:
        extract_chunks_with_metadata(child, code_bytes, file_path, results)

    return results

In [30]:
all_chunks = extract_chunks_with_metadata(tree.root_node, code_bytes, file_path="my_script.py")


In [31]:
for item in all_chunks:
    meta = item['metadata']
    print(f"--- NODE FOUND ---")
    print(f"Code: {item['code']}")
    print(f"Type: {meta['node_type']}")
    print(f"Name: {meta['name']}")
    print(f"Signature: {meta['signature']}")
    print(f"Dependencies: {meta['dependencies']}")
    print(f"Lines: {meta['start_line']} to {meta['end_line']}")
    # print(f"Code Snippet:\n{item['code']}") # Uncomment to see the actual code
    print("\n")

--- NODE FOUND ---
Code: function formReducer(state, action) {
  switch (action.type) {
    case "UPDATE_FIELD":
      return {
        ...state,
        [action.field]: action.value,
      };
    case "RESET":
      return initialState;
    default:
      return state;
  }
}
Type: function_declaration
Name: formReducer
Signature: function formReducer(state, action)
Dependencies: []
Lines: 35 to 47


--- NODE FOUND ---
Code: ({ children }) => {
  const [companyDetails, setcompanyDetails] = useState(null);
  const [companyList, setcompanyList] = useState(null);
  const [isAuthenticating, setisAuthenticating] = useState(true);
  const [errors, setErrors] = useState({});
  const [companyForm, companyFormDispatch] = useReducer(
    formReducer,
    initialState
  );
  const { userDetails } = useContext(UserContext);
  const navigate = useNavigate();

  // function to get company details
  const getCompanyDetails = useCallback(
    async (setisLoading = () => {}) => {
      const companyId 

In [35]:


def extract_chunks_with_metadata(node, code_bytes, file_path="unknown.py", results=None):
    if results is None:
        results = []

    # Get the right targets for the file extension
    target_types = [
        "function_definition", "function_declaration", "method_definition",
        "class_definition", "class_declaration", "arrow_function", 
        "if_statement", "for_statement", "while_statement", "try_statement", "jsx_element"
    ]


    # We always allow comments to be visited, but they are usually handled by the 'look-back' logic
    if node.is_named and node.type in target_types:
        chunk_code = code_bytes[node.start_byte:node.end_byte].decode("utf8", errors="ignore")
        start_line = node.start_point[0] + 1

        # --- 1. NEW: COMMENT EXTRACTION (Look-Back Logic) ---
        comment_text = ""
        prev_node = node.prev_sibling
        
        # Look at siblings immediately above the node. 
        # We check for 'comment' or 'expression_statement' (for Python docstrings)
        while prev_node and prev_node.type in ["comment", "line_comment", "block_comment", "expression_statement"]:
            # Capture the comment text
            text = code_bytes[prev_node.start_byte:prev_node.end_byte].decode("utf8", errors="ignore").strip()
            
            # Special check for Python docstrings (which are expression_statements containing a string)
            if prev_node.type == "expression_statement":
                if "string" not in prev_node.children[0].type:
                    break # Not a docstring
            
            comment_text = text + "\n" + comment_text
            prev_node = prev_node.prev_sibling

        # --- 2. SMART NAMING ---
        name = "anonymous"
        name_node = node.child_by_field_name("name")
        if not name_node and node.type == "arrow_function":
            parent = node.parent
            if parent and parent.type == "variable_declarator":
                name_node = parent.child_by_field_name("name")
        
        if name_node:
            name = code_bytes[name_node.start_byte:name_node.end_byte].decode("utf8", errors="ignore")
        else:
            name = f"{node.type}_at_line_{start_line}"

        # --- 3. SIGNATURE & DEPENDENCIES ---
        body_node = node.child_by_field_name("body")
        signature = code_bytes[node.start_byte:body_node.start_byte].decode("utf8").strip() if body_node else chunk_code.split('\n')[0]

        def find_calls(n, calls_list):
            for child in n.children:
                if child.type == "call_expression":
                    call_id = child.child_by_field_name("function") or (child.children[0] if child.children else None)
                    if call_id:
                        calls_list.append(code_bytes[call_id.start_byte:call_id.end_byte].decode("utf8", errors="ignore"))
                find_calls(child, calls_list)

        calls = []
        find_calls(node, calls)

        # Build final object
        results.append({
            "code": chunk_code,
            "metadata": {
                "name": name,
                "node_type": node.type,
                "comment": comment_text.strip(), # New Metadata Field
                "file_path": file_path,
                "signature": signature,
                "dependencies": list(set(calls)),
                "start_line": start_line,
                "end_line": node.end_point[0] + 1
            }
        })

    # Recursive traversal
    for child in node.children:
        extract_chunks_with_metadata(child, code_bytes, file_path, results)

    return results

In [36]:
all_chunks = extract_chunks_with_metadata(tree.root_node, code_bytes, file_path="my_script.py")


In [37]:
for item in all_chunks:
    meta = item['metadata']
    print(f"--- NODE FOUND ---")
    print(f"comment: {meta['comment']}")
    print(f"Type: {meta['node_type']}")
    print(f"Name: {meta['name']}")
    print(f"Signature: {meta['signature']}")
    print(f"Dependencies: {meta['dependencies']}")
    print(f"Lines: {meta['start_line']} to {meta['end_line']}")
    # print(f"Code Snippet:\n{item['code']}") # Uncomment to see the actual code
    print("\n")

--- NODE FOUND ---
comment: 
Type: function_declaration
Name: formReducer
Signature: function formReducer(state, action)
Dependencies: []
Lines: 35 to 47


--- NODE FOUND ---
comment: 
Type: arrow_function
Name: CompanyContextProvider
Signature: ({ children }) =>
Dependencies: ['res.data.status.toLowerCase', 'useContext', 'companyFormDispatch', 'navigate', 'res.data.data.slice', 'useReducer', 'getCompanyList', 'res.data.data.filter', 'uploadFile', 'Object.keys', 'setisAuthenticating', 'useNavigate', 'showToast', 'res.data.error_message?.toLowerCase', 'setErrors', 'useCallback', 'localStorage.getItem', 'setcompanyList', 'localStorage.getItem("companyid").toString', 'validateFields', 'window.location.reload', 'getCompanyDetails', 'setcompanyDetails', 'res.data.error_message.toLowerCase', 'axios.post', 'useEffect', 'axios.get', 'console.log', 'localStorage.setItem', 'useState', 'setisLoading']
Lines: 49 to 412


--- NODE FOUND ---
comment: 
Type: arrow_function
Name: arrow_function_at_lin